# Adaptive NeSy-Gen: IU-Xray and MIMIC-CXR experiments

This notebook uses the existing Drive manifests, PrimeKG caches, and MedSigLIP indices. It supports frozen-vision CheXagent QLoRA and MedGemma with no task-specific fine-tuning, followed by adaptive claim verification and replayed ablations. Use a GPU runtime.

**Environment boundary:** CheXagent's released code pins `transformers==4.40.0`; MedGemma needs a newer Transformers release. Run those sections in separate Colab runtimes and save outputs to Drive. Do not install both extras in one runtime.

In [ ]:
from google.colab import drive
from tqdm.auto import tqdm
cell_progress = tqdm(total=3, desc='Cell 1/8 · Mount Drive and configure', unit='step', dynamic_ncols=True)
drive.mount('/content/drive')
cell_progress.update()

RUN_DATASET = 'iuxray_official'  # 'iuxray_official' or 'mimic_aug'
REPO_URL = 'https://github.com/FaezehMillerAI/NesyGENAAAI.git'
REPO_DIR = '/content/adaptive-nesy-gen'
OUTPUT_ROOT = f'/content/drive/MyDrive/aaai_2026_experiments/adaptive_nesy_gen/{RUN_DATASET}'
cell_progress.update(2)
cell_progress.close()


In [ ]:
import importlib, os, pathlib, subprocess, sys
from tqdm.auto import tqdm
cell_progress = tqdm(total=5, desc='Cell 2/8 · Bootstrap repository', unit='step', dynamic_ncols=True)
repo_path = pathlib.Path(REPO_DIR)
if not repo_path.exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
elif (repo_path / '.git').exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    raise RuntimeError(f'{REPO_DIR} exists but is not the expected Git repository')
cell_progress.update()
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
cell_progress.update()
src_path = str(repo_path / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
for module_name in [name for name in sys.modules if name == 'adaptive_nesy_gen' or name.startswith('adaptive_nesy_gen.')]:
    del sys.modules[module_name]
importlib.invalidate_caches()
cell_progress.update()
import adaptive_nesy_gen
from adaptive_nesy_gen.retrieval import load_manifest as _bootstrap_load_manifest
cell_progress.update()
print('Repository:', repo_path)
print('Adaptive NeSy-Gen imported from:', adaptive_nesy_gen.__file__)
assert pathlib.Path(adaptive_nesy_gen.__file__).resolve().is_relative_to((repo_path / 'src').resolve())
cell_progress.update()
cell_progress.close()


In [ ]:
import json, os, time, numpy as np
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from adaptive_nesy_gen.retrieval import load_manifest
from tqdm.auto import tqdm
cell_progress = tqdm(total=6, desc='Cell 3/8 · Validate data and caches', unit='step', dynamic_ncols=True)

PATHS = json.loads(Path('configs/colab_paths.json').read_text())[RUN_DATASET]
MANIFEST = PATHS['manifest']
RAW_PRIMEKG_DIR = Path('/content/drive/MyDrive/dataverse_files')
RAD_PRIMEKG_DIR = Path(f'/content/drive/MyDrive/primekg_radiology_cache_{RUN_DATASET}')
PRIMEKG_CACHE = str(RAD_PRIMEKG_DIR)
MEDSIGLIP_CACHE = PATHS['medsiglip_cache']
cell_progress.update()
REBUILD_RAD_PRIMEKG = False
if REBUILD_RAD_PRIMEKG:
    subprocess.run([
        sys.executable, 'scripts/build_radiology_primekg.py',
        '--primekg-dir', str(RAW_PRIMEKG_DIR), '--manifest', MANIFEST,
        '--output-dir', str(RAD_PRIMEKG_DIR), '--hops', '1',
        '--seed-split', 'train',
    ], check=True)
cell_progress.update()
studies = load_manifest(MANIFEST, redact_splits={'test'})
counts = {split: sum(s.split == split for s in studies) for split in ('train', 'val', 'test')}
assert counts['train'] and counts['val'] and counts['test']
cell_progress.update()
image_paths = [s.image_path for s in studies]
with ThreadPoolExecutor(max_workers=16) as pool:
    exists = list(tqdm(
        pool.map(lambda path: Path(path).exists(), image_paths),
        total=len(image_paths), desc='Checking manifest images', unit='image',
        dynamic_ncols=True, leave=False,
    ))
missing = [path for path, is_present in zip(image_paths, exists) if not is_present]
assert not missing, f'{len(missing)} manifest images are missing; first={missing[:3]}'
cell_progress.update()
AUTO_REBUILD_MEDSIGLIP_CACHE = True
MEDSIGLIP_BATCH_SIZE = 16
def inspect_medsiglip_cache(path):
    if not Path(path).exists():
        return None, None
    with np.load(path, allow_pickle=False) as loaded:
        key = next((k for k in ('embeddings', 'image_embeddings', 'features') if k in loaded), None)
        return key, tuple(loaded[key].shape) if key else None
embedding_key, cache_shape = inspect_medsiglip_cache(MEDSIGLIP_CACHE)
cache_aligned = bool(cache_shape) and len(cache_shape) == 2 and counts['train'] in cache_shape
if not cache_aligned:
    message = f'MedSigLIP cache {cache_shape} does not cover {counts["train"]} training examples.'
    if not AUTO_REBUILD_MEDSIGLIP_CACHE:
        raise RuntimeError(message + ' Set AUTO_REBUILD_MEDSIGLIP_CACHE=True.')
    print(message, 'Rebuilding the complete train-only cache now...')
    print('MedSigLIP access terms: https://huggingface.co/google/medsiglip-448')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[medsiglip]'], check=True)
    from getpass import getpass
    from huggingface_hub import hf_hub_download
    def prompt_hf_read_token(model_name):
        token = getpass(f'Paste the Hugging Face read token for {model_name} (input hidden): ').strip()
        if not token:
            raise RuntimeError('No Hugging Face token was entered.')
        os.environ['HF_TOKEN'] = token  # Explicitly override stale cached credentials.
        return token
    try:
        from huggingface_hub import get_token
        hf_token = get_token()
    except ImportError:
        from huggingface_hub import HfFolder
        hf_token = HfFolder.get_token()
    if not hf_token:
        hf_token = prompt_hf_read_token('google/medsiglip-448')
    try:
        hf_hub_download('google/medsiglip-448', 'config.json', token=hf_token)
    except Exception as exc:
        status_code = getattr(getattr(exc, 'response', None), 'status_code', None)
        if status_code not in (401, 403):
            raise
        print('The cached Colab credential was rejected. Please paste a fresh read token from the approved account.')
        hf_token = prompt_hf_read_token('google/medsiglip-448')
        try:
            hf_hub_download('google/medsiglip-448', 'config.json', token=hf_token)
        except Exception as retry_exc:
            raise RuntimeError(
                'The replacement token is also denied. It must belong to the Hugging Face '
                'account that accepted MedSigLIP terms and include read access to this gated repo.'
            ) from retry_exc
    rebuild_cmd = [
        sys.executable, 'scripts/build_medsiglip_index.py',
        '--manifest', MANIFEST, '--output', MEDSIGLIP_CACHE,
        '--batch-size', str(MEDSIGLIP_BATCH_SIZE),
        '--progress-file', '/tmp/medsiglip_rebuild_progress.json',
    ]
    progress_file = Path('/tmp/medsiglip_rebuild_progress.json')
    progress_file.unlink(missing_ok=True)
    rebuild_bar = tqdm(
        total=counts['train'], desc='Rebuilding complete train-only cache',
        unit='image', dynamic_ncols=True, leave=True,
    )
    rebuild = subprocess.Popen(rebuild_cmd)
    displayed = 0
    while rebuild.poll() is None:
        if progress_file.exists():
            try:
                status = json.loads(progress_file.read_text(encoding='utf-8'))
                completed = min(int(status.get('completed', 0)), counts['train'])
                rebuild_bar.set_postfix_str(status.get('stage', 'Working'))
                rebuild_bar.update(max(0, completed - displayed))
                displayed = max(displayed, completed)
            except (OSError, ValueError, json.JSONDecodeError):
                pass
        rebuild_bar.refresh()
        time.sleep(0.5)
    if rebuild.returncode == 0:
        rebuild_bar.update(counts['train'] - displayed)
        rebuild_bar.set_postfix_str('Complete')
    else:
        rebuild_bar.set_postfix_str(f'Failed (exit {rebuild.returncode})')
    rebuild_bar.close()
    if rebuild.returncode:
        raise RuntimeError(
            f'MedSigLIP rebuild failed with exit code {rebuild.returncode}. '
            'Scroll to the first traceback above this message for the underlying error.'
        )
    embedding_key, cache_shape = inspect_medsiglip_cache(MEDSIGLIP_CACHE)
cache_aligned = bool(cache_shape) and len(cache_shape) == 2 and counts['train'] in cache_shape
assert cache_aligned, (cache_shape, counts)
index_shape = cache_shape if cache_shape[0] == counts['train'] else (cache_shape[1], cache_shape[0])
if cache_shape[0] != counts['train']:
    print('Detected legacy transposed cache; it will be transposed while loading:', cache_shape)
cell_progress.update()
assert RAD_PRIMEKG_DIR == Path(PATHS['primekg_cache'])
for required in ('kg.csv', 'nodes.csv', 'radiology_primekg_summary.json'):
    assert (RAD_PRIMEKG_DIR / required).exists(), RAD_PRIMEKG_DIR / required
print({'splits': counts, 'index_shape': index_shape, 'primekg': PRIMEKG_CACHE})
cell_progress.update()
cell_progress.close()


In [ ]:
# Fast contract checks before any expensive model load.
from tqdm.auto import tqdm
cell_progress = tqdm(total=2, desc='Cell 4/8 · Run smoke checks', unit='check', dynamic_ncols=True)
subprocess.run(['pytest', '-q'], check=True)
cell_progress.update()
subprocess.run(['adaptive-nesy-gen', 'demo', '--output', '/content/demo_trace.json'], check=True)
cell_progress.update()
cell_progress.close()


## A. Efficient CheXagent QLoRA (dedicated runtime)

The script uses NF4 QLoRA only on decoder modules, keeps the entire visual encoder frozen in BF16, applies 512×512 augmentation without flips, masks prompt tokens from the loss, uses only train/val, and inserts the first three of five unique MedSigLIP neighbours into 50% of training prompts. Set `RUN_QLORA=True` when ready.

In [ ]:
from tqdm.auto import tqdm
cell_progress = tqdm(total=3, desc='Cell 5/8 · CheXagent QLoRA', unit='step', dynamic_ncols=True)
RUN_QLORA = True
MAX_STEPS = 500  # 20 for a pipeline check; 500+ for the planned run
CHEX_OUTPUT = f'{OUTPUT_ROOT}/chexagent_qlora'
cell_progress.update()
if RUN_QLORA:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', '-e', '.[chexagent]'], check=True)
    cell_progress.update()
    version_probe = subprocess.run(
        [sys.executable, '-c', 'import peft, torch, transformers; print({"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__, "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "bf16": torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False})'],
        text=True, capture_output=True, check=True,
    )
    print('CheXagent environment:', version_probe.stdout.strip())
    import torch
    gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
    CHEX_BATCH_SIZE = 4 if gpu_memory_gb >= 40 else 1
    CHEX_GRADIENT_ACCUMULATION = 4 if CHEX_BATCH_SIZE == 4 else 16
    print({'batch_size': CHEX_BATCH_SIZE, 'gradient_accumulation': CHEX_GRADIENT_ACCUMULATION, 'effective_batch': CHEX_BATCH_SIZE * CHEX_GRADIENT_ACCUMULATION})
    cmd = [
        sys.executable, 'scripts/train_chexagent_qlora.py',
        '--manifest', MANIFEST, '--medsiglip-cache', MEDSIGLIP_CACHE,
        '--output-dir', CHEX_OUTPUT, '--max-steps', str(MAX_STEPS),
        '--batch-size', str(CHEX_BATCH_SIZE),
        '--gradient-accumulation', str(CHEX_GRADIENT_ACCUMULATION),
    ]
    training_error_log = f'{CHEX_OUTPUT}/training_error.log'
    Path(CHEX_OUTPUT).mkdir(parents=True, exist_ok=True)
    Path(training_error_log).unlink(missing_ok=True)
    training_env = os.environ.copy()
    training_env['ADAPTIVE_NESY_TRAIN_ERROR_LOG'] = training_error_log
    training = subprocess.run(cmd, env=training_env)
    if training.returncode:
        details = Path(training_error_log).read_text(encoding='utf-8') if Path(training_error_log).exists() else 'No child traceback was written.'
        raise RuntimeError(
            f'CheXagent QLoRA failed with exit code {training.returncode}.\n\n{details}'
        )
    cell_progress.update()
else:
    cell_progress.update(2)
    print('QLoRA skipped. Set RUN_QLORA=True to train.')
cell_progress.close()


## B. Generate a test subset once

Choose one backend per fresh runtime. For CheXagent install `.[chexagent]` and provide its adapter. For MedGemma install `.[medgemma]`, accept the gated Hugging Face terms, and authenticate. On MIMIC-CXR describe MedGemma as **no task-specific fine-tuning**, not strict unseen-data zero-shot. `LIMIT=None` runs the complete official test split.

In [ ]:
from tqdm.auto import tqdm
cell_progress = tqdm(total=4, desc='Cell 6/8 · Generate reports', unit='step', dynamic_ncols=True)
BACKEND = 'chexagent'  # trained default; alternatives: 'medgemma' or 'retrieval'
DRAFTING_MODE = 'few-shot'  # 'zero-shot' or 'few-shot'
LIMIT = None  # None runs every row in the official test split
cell_progress.update()
if BACKEND == 'medgemma':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[medgemma]'], check=True)
    from getpass import getpass
    from huggingface_hub import hf_hub_download
    def prompt_hf_read_token(model_name):
        token = getpass(f'Paste the Hugging Face read token for {model_name} (input hidden): ').strip()
        if not token:
            raise RuntimeError('No Hugging Face token was entered.')
        os.environ['HF_TOKEN'] = token
        return token
    try:
        from huggingface_hub import get_token
        hf_token = get_token()
    except ImportError:
        from huggingface_hub import HfFolder
        hf_token = HfFolder.get_token()
    if not hf_token:
        hf_token = prompt_hf_read_token('google/medgemma-4b-it')
    print('Checking MedGemma access: https://huggingface.co/google/medgemma-4b-it')
    try:
        hf_hub_download('google/medgemma-4b-it', 'config.json', token=hf_token)
    except Exception as exc:
        status_code = getattr(getattr(exc, 'response', None), 'status_code', None)
        if status_code not in (401, 403):
            raise
        print('The cached Colab credential was rejected. Please paste a fresh read token from the approved account.')
        hf_token = prompt_hf_read_token('google/medgemma-4b-it')
        try:
            hf_hub_download('google/medgemma-4b-it', 'config.json', token=hf_token)
        except Exception as retry_exc:
            raise RuntimeError(
                'The replacement token is also denied. It must belong to the Hugging Face '
                'account that accepted MedGemma terms and include read access to this gated repo.'
            ) from retry_exc
elif BACKEND == 'chexagent':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[chexagent]'], check=True)
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[medgemma]'], check=True)
cell_progress.update()

draft_file = f'{OUTPUT_ROOT}/{BACKEND}_{DRAFTING_MODE}_full_adaptive.jsonl'
cmd = [
    sys.executable, 'scripts/run_experiments.py', '--manifest', MANIFEST,
    '--medsiglip-cache', MEDSIGLIP_CACHE, '--primekg-cache', PRIMEKG_CACHE,
    '--output', draft_file, '--backend', BACKEND, '--drafting-mode', DRAFTING_MODE,
    '--ablation', 'full_adaptive', '--top-k', '5',
]
if LIMIT: cmd += ['--limit', str(LIMIT)]
if BACKEND == 'chexagent': cmd += ['--adapter', f'{CHEX_OUTPUT}/adapter']
if BACKEND == 'chexagent': assert Path(f'{CHEX_OUTPUT}/adapter').exists(), 'Run Cell 5 first'
cell_progress.update()
error_log = f'{draft_file}.error.log'
Path(error_log).unlink(missing_ok=True)
generation_env = os.environ.copy()
generation_env['ADAPTIVE_NESY_ERROR_LOG'] = error_log
generation = subprocess.run(cmd, env=generation_env)
if generation.returncode:
    details = Path(error_log).read_text(encoding='utf-8') if Path(error_log).exists() else 'No child traceback was written.'
    raise RuntimeError(
        f'Generation failed with exit code {generation.returncode}.\n\n{details}'
    )
with open(draft_file, encoding='utf-8') as handle:
    generated_records = [json.loads(line) for line in handle if line.strip()]
assert len(generated_records) == counts['test'], (len(generated_records), counts['test'])
assert all('reference' not in record for record in generated_records)
assert all(not record['test_reference_consumed_during_inference'] for record in generated_records)
cell_progress.update()
cell_progress.close()
print(draft_file)


## C. Replay verifier ablations

Generation is the expensive and stochastic comparison unit. These runs reuse each saved raw draft and rerun only retrieval, linking, graph constraints, gate, and revision. Add report-level verification and shuffled/relation-ablated PrimeKG caches as separate controls.

In [ ]:
from tqdm.auto import tqdm
cell_progress = tqdm(total=3, desc='Cell 7/8 · Replay ablations', unit='step', dynamic_ncols=True)
suite_dir = f'{OUTPUT_ROOT}/replay_{BACKEND}_{DRAFTING_MODE}_suite'
cmd = [
    sys.executable, 'scripts/run_experiments.py', '--manifest', MANIFEST,
    '--medsiglip-cache', MEDSIGLIP_CACHE, '--primekg-cache', PRIMEKG_CACHE,
    '--output', suite_dir, '--backend', 'replay', '--drafts-jsonl', draft_file,
    '--drafting-mode', DRAFTING_MODE, '--suite', '--top-k', '5',
]
if LIMIT: cmd += ['--limit', str(LIMIT)]
cell_progress.update()
subprocess.run(cmd, check=True)
cell_progress.update()
ablation_files = {p.stem: str(p) for p in Path(suite_dir).glob('*.jsonl')}
print('Completed:', sorted(ablation_files))
cell_progress.update()
cell_progress.close()


In [ ]:
# Test references are first loaded inside this post-inference evaluator.
import json, shutil
from tqdm.auto import tqdm
cell_progress = tqdm(total=5, desc='Cell 8/8 · Publication evaluation', unit='step', dynamic_ncols=True)
if shutil.which('java') is None:
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jre-headless'], check=True)
cell_progress.update()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[eval]'], check=True)
import pandas as pd
cell_progress.update()
PUBLICATION_DIR = f'{OUTPUT_ROOT}/publication_evaluation'
evaluation_cmd = [
    sys.executable, 'scripts/evaluate_publication.py',
    '--manifest', MANIFEST, '--predictions-dir', suite_dir,
    '--primekg-cache', PRIMEKG_CACHE, '--output-dir', PUBLICATION_DIR,
    '--baseline', 'full_adaptive', '--bootstrap-samples', '2000',
    '--expert-sample-size', '100', '--seed', '13',
]
cell_progress.update()
subprocess.run(evaluation_cmd, check=True)
cell_progress.update()
publication_path = Path(PUBLICATION_DIR) / 'publication_metrics.json'
publication = json.loads(publication_path.read_text())
assert all(method['integrity']['passed'] for method in publication['methods'].values())
summary_df = pd.DataFrame({
    name: {**result['official_metrics'], **result['pipeline'], **result['resources']}
    for name, result in publication['methods'].items()
}).T
display(summary_df)
BLEU1_TARGET = 0.50
best_bleu1_method = summary_df['bleu_1'].astype(float).idxmax()
best_bleu1 = float(summary_df.loc[best_bleu1_method, 'bleu_1'])
print({'bleu_1_target': BLEU1_TARGET, 'best_method': best_bleu1_method, 'best_bleu_1': best_bleu1, 'target_met': best_bleu1 >= BLEU1_TARGET})
cell_progress.update()
cell_progress.close()
print('Saved:', publication_path)
print('Expert review:', publication['expert_review_status'])
print(publication['claim_policy'])


## Publication checklist

The cells above execute the full official test split, enforce a post-inference reference firewall, collect all timing/resource/routing fields, run official COCO BLEU-1–4/ROUGE-L/METEOR/CIDEr plus F1CheXbert and F1RadGraph, audit retrieval/leakage/diversity/linker coverage, and compute paired bootstrap confidence intervals with Holm correction.

The final cell also creates blinded `expert_review_packet.csv`, its separate blinding key, `expert_review_protocol.md`, and `linker_expert_review_packet.csv`. Human expert review cannot be automated: `publication_metrics.json` remains `PENDING_HUMAN_REVIEW`, and hallucination-reduction claims are blocked until completed reviews are adjudicated. LTN values are implemented constraint-satisfaction scores; traces are procedural records, not complete causal explanations.